# Loan Dataset - Data Understanding and Preprocessing
This notebook provides a clean coursework workflow for understanding the loan approval dataset, handling missing values, encoding categorical variables, preparing classification and regression datasets, and scaling features.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Configure plots for coursework presentation
sns.set(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)


## 1. Import Libraries
Import the required libraries for data analysis, visualization, and preprocessing.

In [ ]:
# Load the dataset into a pandas DataFrame
file_path = 'loan_approval_data.csv'
df = pd.read_csv(file_path)

# Display the first rows of the dataset
print('=== First 5 rows of the dataset ===')
print(df.head())


## 2. Load Dataset
Read the CSV file and inspect the dataset with head, info, and describe.

In [ ]:
# Show DataFrame information and descriptive statistics
print('\n=== Dataset Information ===')
print(df.info())

print('\n=== Summary Statistics for Numerical Columns ===')
print(df.describe().transpose())


## 3. Data Types
Inspect column types and separate numerical and categorical features.

In [ ]:
# Print column data types and separate numerical and categorical columns
print('=== Column Data Types ===')
print(df.dtypes)

# Separate columns into numerical and categorical lists
numerical_columns = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = df.select_dtypes(include=['object', 'str']).columns.tolist()

print('\n=== Numerical (Continuous) Columns ===')
print(numerical_columns)

print('\n=== Categorical (Nominal) Columns ===')
print(categorical_columns)


In [ ]:
# Check missing values before handling
missing_before = df.isnull().sum()
print('=== Missing Values BEFORE Handling ===')
missing_counts = missing_before[missing_before > 0]
if len(missing_counts) > 0:
    print(missing_counts)
else:
    print('No missing values found')

# Handle missing values for numerical features with median
for col in numerical_columns:
    if df[col].isnull().any():
        median_value = df[col].median()
        df.loc[:, col] = df[col].fillna(median_value)

# Handle missing values for categorical features with mode
for col in categorical_columns:
    if df[col].isnull().any():
        mode_value = df[col].mode()[0]
        df.loc[:, col] = df[col].fillna(mode_value)

# Verify missing values after handling
missing_after = df.isnull().sum()
print('\n=== Missing Values AFTER Handling ===')
missing_counts_after = missing_after[missing_after > 0]
if len(missing_counts_after) > 0:
    print(missing_counts_after)
else:
    print('No missing values found')


## 4. Missing Values
Show missing values before and after imputation using median for numerical and mode for categorical features.

In [ ]:
# Remove irrelevant ID columns if they exist
id_columns = [col for col in df.columns if col.lower() == 'id' or col.lower().endswith('_id')]
print('=== ID Columns to Drop ===')
print(id_columns if id_columns else 'No ID columns found')

# Drop ID columns
if id_columns:
    df.drop(columns=id_columns, inplace=True)
    print('Columns dropped successfully')

# Update feature lists after dropping irrelevant columns
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'\n=== Updated Column Lists ===')
print(f'Numerical: {numerical_columns}')
print(f'Categorical: {categorical_columns}')


## 5. Remove Irrelevant Columns
Drop ID columns if they exist before continuing with preprocessing.

## 6. Encode Categorical Variables
Use LabelEncoder to convert categorical features into numerical format. Note that this method may introduce ordinal bias.

In [ ]:
# Encode categorical variables using LabelEncoder
# Note: Label Encoding may introduce ordinal bias
encoder_dict = {}

for col in categorical_columns:
    encoder_dict[col] = LabelEncoder()
    df[col] = encoder_dict[col].fit_transform(df[col].astype(str))

print('=== Categorical Variables Encoded ===')
print(f'Columns encoded: {categorical_columns}')
print('\nWarning: Label Encoding may introduce ordinal bias. Consider using OneHotEncoder for tree-based models.')


## 7. Create Classification Dataset
Create the classification dataset with Loan_Approval_Status (0=Approved, 1=Rejected) as the target variable.

In [ ]:
# --- Print all available columns ---
print("Available columns:")
print(df.columns.tolist())

# --- Auto-detect classification target (loan approval status) ---
classification_target = None
for col in df.columns:
    if 'approval' in col.lower():
        classification_target = col
        break

# --- Auto-detect regression target (maximum loan amount) ---
regression_target = None
for col in df.columns:
    if 'loan' in col.lower() and 'amount' in col.lower():
        regression_target = col
        break

# --- Print detected targets ---
print(f"Detected classification target: {classification_target}")
print(f"Detected regression target: {regression_target}")

# --- Safety check for targets ---
if classification_target is None or regression_target is None:
    print("ERROR: Could not auto-detect targets. Available columns:")
    print(df.columns.tolist())
else:
    # --- Create classification dataset ---
    X_classification = df.drop(columns=[classification_target])
    y_classification = df[classification_target]

    print('=== Classification Dataset Created ===')
    print(f'X_classification shape: {X_classification.shape}')
    print(f'y_classification shape: {y_classification.shape}')

    print('\n=== Target Value Counts ===')
    print(y_classification.value_counts(dropna=False).sort_index())
    print('\nMapping: 0 = Approved, 1 = Rejected')


In [ ]:
# Plot the target distribution for loan approval status
print('=== Target Distribution Plot ===')
plt.figure(figsize=(8, 5))
sns.countplot(x=y_classification, palette='Set2')
plt.title('Loan Approval Status Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Loan Approval Status (0 = Approved, 1 = Rejected)', fontsize=11)
plt.ylabel('Count', fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary statistics
print(f'Total Applications: {len(y_classification)}')
print(f'Approved: {(y_classification == 0).sum()} ({(y_classification == 0).sum() / len(y_classification) * 100:.2f}%)')
print(f'Rejected: {(y_classification == 1).sum()} ({(y_classification == 1).sum() / len(y_classification) * 100:.2f}%)')


## 8. Plot Target Distribution
Visualize the distribution of the loan approval status to understand class balance in the dataset.

In [ ]:
# --- Create regression dataset for maximum loan amount using only approved loans ---
# (This cell assumes classification_target and regression_target are already detected)

if classification_target is None or regression_target is None:
    print("ERROR: Targets not detected. Cannot create regression dataset.")
    print("Available columns:", df.columns.tolist())
else:
    approved_loans_mask = df[classification_target] == 0
    approved_loans = df[approved_loans_mask].copy()

    X_regression = approved_loans.drop(columns=[regression_target, classification_target])
    y_regression = approved_loans[regression_target]

    print('=== Regression Dataset Created (Approved Loans Only) ===')
    print(f'X_regression shape: {X_regression.shape}')
    print(f'y_regression shape: {y_regression.shape}')

    print(f'\n=== Regression Target Statistics ===')
    print(y_regression.describe())


## 9. Create Regression Dataset
Create regression dataset for predicting Maximum_Loan_Amount using only approved loans (Approval_Status = 0).

## 10. Feature Scaling
Apply StandardScaler to normalize all features to have mean=0 and standard deviation=1 for both datasets.

In [ ]:
# --- Scale the feature sets using StandardScaler ---
# (This cell assumes X_classification, X_regression exist)
from sklearn.preprocessing import StandardScaler

scaler_classification = StandardScaler()
scaler_regression = StandardScaler()

if 'X_classification' in locals() and X_classification is not None:
    X_classification_scaled = scaler_classification.fit_transform(X_classification)
    print('=== Feature Scaling Complete (Classification) ===')
    print(f'X_classification_scaled shape: {X_classification_scaled.shape}')
    print(f'X_classification_scaled mean (should be ~0): {X_classification_scaled.mean(axis=0).mean():.6f}')
    print(f'X_classification_scaled std (should be ~1): {X_classification_scaled.std(axis=0).mean():.6f}')
else:
    print("ERROR: X_classification not available for scaling.")

if 'X_regression' in locals() and X_regression is not None:
    X_regression_scaled = scaler_regression.fit_transform(X_regression)
    print('=== Feature Scaling Complete (Regression) ===')
    print(f'X_regression_scaled shape: {X_regression_scaled.shape}')
    print(f'X_regression_scaled mean (should be ~0): {X_regression_scaled.mean(axis=0).mean():.6f}')
    print(f'X_regression_scaled std (should be ~1): {X_regression_scaled.std(axis=0).mean():.6f}')
else:
    print("ERROR: X_regression not available for scaling.")


In [ ]:
# --- Convert scaled arrays to DataFrames to preserve feature names ---
if 'X_classification_scaled' in locals() and 'X_classification' in locals():
    X_classification_scaled = pd.DataFrame(
        X_classification_scaled,
        columns=X_classification.columns
    )
    print(type(X_classification_scaled))

if 'X_regression_scaled' in locals() and 'X_regression' in locals():
    X_regression_scaled = pd.DataFrame(
        X_regression_scaled,
        columns=X_regression.columns
    )
    print(type(X_regression_scaled))

# --- Ensuring target variable is numeric for model compatibility ---
if 'y_classification' in locals():
    y_classification = y_classification.astype(int)
    print(type(y_classification))


In [ ]:
# Final summary of all prepared datasets
print('\n' + '='*60)
print('PREPROCESSING COMPLETE - FINAL SUMMARY')
print('='*60)

print('\n=== Classification Dataset ===')
print(f'X_classification shape: {X_classification.shape}')
print(f'y_classification shape: {y_classification.shape}')
print(f'X_classification_scaled shape: {X_classification_scaled.shape}')

print('\n=== Regression Dataset ===')
print(f'X_regression shape: {X_regression.shape}')
print(f'y_regression shape: {y_regression.shape}')
print(f'X_regression_scaled shape: {X_regression_scaled.shape}')

print('\n=== Variables Ready for Model Training ===')
print('✓ X_classification_scaled - Classification features (scaled)')
print('✓ y_classification - Classification target (0=Approved, 1=Rejected)')
print('✓ X_regression_scaled - Regression features from approved loans (scaled)')
print('✓ y_regression - Regression target (Maximum Loan Amount)')
print('\nAll variables are ready for import into Notebook 2')
print('='*60)
